In [3]:
!pip install swig
!pip install gymnasium[box2d] moviepy

import gymnasium as gym
import numpy as np
import tensorflow as tf
import moviepy.editor as mpy
import os

env = gym.make('CartPole-v1', render_mode="rgb_array_list")

max_steps = 50_000
step = 0
lr = 0.005
gamma = 0.9999


# //////////////
# estados (4):
#       posición del carro,
#       velocidad del carro,
#       ángulo del poste,
#       velocidad angular del poste
#
# salidas (2): izq o der
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(4,)),
    tf.keras.layers.Dense(int(env.action_space.n), activation='softmax') #  int
])

optimizer = tf.keras.optimizers.Adam(learning_rate=lr)

# entrenamiento
while step <= max_steps:
    obs, _ = env.reset()
    done = False
    Actions, States, Rewards = [], [], []

    while not done:
        # //////////
        # eleccion de una accion
        obs_tensor = tf.convert_to_tensor([obs], dtype=tf.float32)
        # /////////
        # le paso el estado actual a la red
        probs = model(obs_tensor)
        # /////////
        # se elige una accion aleatoria
        action = tf.random.categorical(tf.math.log(probs), 1)[0, 0].numpy()

        obs_, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        Actions.append(action)
        States.append(obs)
        Rewards.append(reward)

        obs = obs_

        step += 1

    # calculo retornos descontados
    DiscountedReturns = []
    for t in range(len(Rewards)):
        G = 0.0
        for k, r in enumerate(Rewards[t:]):
            G += (gamma**k) * r
        DiscountedReturns.append(G)

    # actualiz. modelo
    for state, action, G in zip(States, Actions, DiscountedReturns):
        with tf.GradientTape() as tape:
            state_tensor = tf.convert_to_tensor([state], dtype=tf.float32)
            probs = model(state_tensor, training=True)
            action_prob = tf.gather(probs[0], action)
            log_prob = tf.math.log(action_prob + 1e-8)  #
            loss = -log_prob * G

        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

# test
print("\nGenerando video...")

video_folder = "./cartpole_videos/"
os.makedirs(video_folder, exist_ok=True)

for episode in range(1):
    obs, _ = env.reset()
    done = False
    frames = []
    Rewards = []

    while not done:
        obs_tensor = tf.convert_to_tensor([obs], dtype=tf.float32)
        probs = model(obs_tensor)
        action = tf.random.categorical(tf.math.log(probs), 1)[0, 0].numpy()

        obs_, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        Rewards.append(reward)
        obs = obs_

    frames = env.render()

# guarda video
video_path = os.path.join(video_folder, "cartpole_run.mp4")
clip = mpy.ImageSequenceClip(frames, fps=30)
clip.write_videofile(video_path)

print(f"Video guardado en {video_path}")

env.close()



Generando video...
Moviepy - Building video ./cartpole_videos/cartpole_run.mp4.
Moviepy - Writing video ./cartpole_videos/cartpole_run.mp4



Moviepy - Done !
Moviepy - video ready ./cartpole_videos/cartpole_run.mp4
Video guardado en ./cartpole_videos/cartpole_run.mp4
